### Structured Output 

Models can be requested to provide their response in a format matching a given schema.This is useful for ensuring the output can be easily parsed and used in subsequent processing.LangChain supports multiple schema types and methods for enforcing structured output.

### 1 . Pydantic

Pydantic models provide the richest feature set with field validation,descriptions,and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")


In [6]:

from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year : int = Field(description="The year movie was released")
    director: str=Field(description="The director of the movie")
    rating :float = Field(description="the Movie's rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000225E743CD70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000225E743DA90>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The tit

In [4]:
model.invoke("Provide details about inception")

AIMessage(content='## “Inception” (2010) – A Comprehensive Overview\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Director / Writer** | Christopher\u202fNolan |\n| **Release Date** | 16\u202fJuly\u202f2010 (USA) |\n| **Runtime** | 148\u202fminutes |\n| **Genre** | Sci‑Fi Action / Thriller |\n| **Production Companies** | Warner\u202fBros. Pictures, Legendary Pictures, Syncopy |\n| **Budget** | ~$160\u202fmillion |\n| **Box‑Office** | ~$829\u202fmillion worldwide |\n| **Language** | English |\n| **Filming Locations** | United Kingdom (London, Oxford), United States (Los Angeles, San Francisco), Morocco (Ouarzazate), Japan (Tokyo) |\n| **Music** | Hans\u202fZimmer (score) |\n| **Cinematography** | Wally\u202fPfister |\n| **Editing** | Lee\u202fSmith |\n| **Distributor** | Warner\u202fBros. Pictures |\n\n---\n\n### 1. Core Premise\n\n- **The Inception Concept**: In the film’s universe, “inception” is the act of planting a fully formed idea in a target’s subcon

In [5]:
model_with_structure.invoke("Provide details about movie inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

#### Message output alongside Parsed Structure

In [7]:
from urllib3 import response
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """ A movie with deatils as below"""
    title: str = Field(description="The title of the movie")
    year : int = Field(description="The year movie was released")
    director: str=Field(description="The director of the movie")
    rating :float = Field(description="the Movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the function to get details.', 'tool_calls': [{'id': 'fc_9d94e58c-95b0-4e62-857a-3530eae44841', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 167, 'total_tokens': 217, 'completion_time': 0.053380772, 'completion_tokens_details': {'reasoning_tokens': 11}, 'prompt_time': 0.011513908, 'prompt_tokens_details': None, 'queue_time': 0.279738922, 'total_time': 0.06489468}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_66891002f6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05ba5-b9dd-7d22-826f-8d0b56614e65-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 'fc_

### Nested Structure 

In [10]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name :str
    role:str
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]     # since there can be multiple actors (eg.,.Alluarjun and Ramcharan in Yevaduuuuuu)
    genres:list[str]
    budget:float| None = Field(None,description="Budget in  USD")

In [11]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160000000.0)

### 2  . TypedDict

TypedDict provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation.

In [12]:
from typing_extensions import TypedDict,Annotated


In [17]:

class MovieDict(TypedDict):
    """A movie with deatils"""
    title:Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"the year movie was released"]
    director: Annotated[str,...,"the director of the moive "]
    rating:Annotated[float,...,"the Movie's rating out of 10"]

In [15]:
model_with_typedDict = model.with_structured_output(MovieDict)
response = model_with_typedDict.invoke("Provide the details of the movie Incredible HULk")
response 

{'director': 'Joss Whedon',
 'rating': 6.5,
 'title': 'The Incredible Hulk',
 'year': 2008}

### 3 . Data Classes

A Data Class is a class typically containing mainly data, although there aaren't really any restrictions. You create it using the @dataclass decorator.

In [19]:
from dataclasses import dataclass

@dataclass
class ContactInfo:
    name:str
    email:str
    phone:str

How These are implemented 

In [18]:
from langchain.agents import create_agent

from pydantic import BaseModel,Field

from typing_extensions import TypedDict

from dataclasses import dataclass


#using pydantic 
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year : int = Field(description="The year movie was released")
    director: str=Field(description="The director of the movie")
    rating :float = Field(description="the Movie's rating out of 10")


#using typeddict
class MovieDict(TypedDict):
    """A movie with deatils"""
    title:Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"the year movie was released"]
    director: Annotated[str,...,"the director of the moive "]
    rating:Annotated[float,...,"the Movie's rating out of 10"]


#using dataclass
@dataclass
class MovieDataClass:
    title:str
    year:int
    director:str
    rating:float
    

In [27]:
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=Movie   # auto-selects ProviderStrategy
)
result = agent.invoke(
    {
        "messages":[{"role":"user","content":"Tell me about movie Intestellar"}]
    }
)
print("The Pydantic structure output\n")
result["structured_response"]

The Pydantic structure output



Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.6)

In [26]:
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=MovieDict   # auto-selects ProviderStrategy
)
result = agent.invoke(
    {
        "messages":[{"role":"user","content":"Tell me about movie Intestellar"}]
    }
)
print("The TypedDict Structure output\n")
result["structured_response"]

The TypedDict Structure output



{'title': 'Interstellar',
 'year': 2014,
 'director': 'Christopher Nolan',
 'rating': 8.6}

In [25]:
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=MovieDataClass   # auto-selects ProviderStrategy
)
result = agent.invoke(
    {
        "messages":[{"role":"user","content":"Tell me about movie Intestellar"}]
    }
)
print("The data class Structure output\n")
result["structured_response"]

The data class Structure output



MovieDataClass(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.6)